## Limpieza de 'DF_CRUZANDOLAMETA_SUCIO.csv'

Esta es la versión con resultados reales — `Scraper_cruzandolameta.ipynb` ya extrae `classificat_h`/`classificat_d` (participantes por género) de dos formatos de la web ("nuevo": API de rankings; "antiguo": contando filas filtradas por género), a diferencia de una primera versión del scraper que solo sacaba el calendario de eventos sin resultados.

Dos particularidades de esta fuente:
- **`lloc`** viene como `"Municipio (Provincia)"` en una sola cadena — se separa directamente, sin necesidad de la heurística de extracción de xipgroc/ccnorte.
- **`modalitat_nom`** mezcla dos cosas según qué formato usó el scraper para esa carrera: para el formato "nuevo" es la categoría de edad real (p.ej. "ABSOLUTA", "INFANTIL", "ADAPTADO ADULTOS"); para el formato "antiguo" (la mayoría) es solo el texto de distancia del evento entero (p.ej. "10 Km", "Varias"), porque ese método de extracción no distingue categorías. Por eso `publico` tendrá señal real solo en una parte de las filas — el resto, sin marca de edad, cae correctamente en Absoluta/General.

Quedan pendientes de resultados los eventos en formato PDF (`estat_esdeveniment == "error"` con `error_detall` empezando por `pdf_pendent_de_parsejar`) — no están en este notebook.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/cruzandolameta/DF_CRUZANDOLAMETA_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig", low_memory=False)

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (1977, 26)

event_id                             int64
nom_cursa                           object
data                                object
lloc                                object
modalitat_nom                       object
modalitat_codi                      object
estat_esdeveniment                  object
error_detall                        object
total_classificats                   int64
classificat_h                        int64
classificat_d                        int64
classificat_total                    int64
classificat_sexe_desconegut          int64
DNF_h                                int64
DNF_d                                int64
DNF_total                            int64
DSQ_h                                int64
DSQ_d                                int64
DSQ_total                            int64
DNS_h                                int64
DNS_d                                int64
DNS_total                            int64
estat_desconegut_h      

,event_id,nom_cursa,data,lloc,modalitat_nom,modalitat_codi,estat_esdeveniment,error_detall,total_classificats,classificat_h,...,DSQ_h,DSQ_d,DSQ_total,DNS_h,DNS_d,DNS_total,estat_desconegut_h,estat_desconegut_d,estat_desconegut_sexe_desconegut,esport
0,2618,LXXXV TRAVESÍA A NADO PUERTO DE MOTRIL,2026-08-15,Motril (Granada),PRUEBA ABSOLUTA,lxxxv-travesia-a-nado-puerto-de-motril-prueba-...,ok,NaN,194,149,...,0,0,0,0,0,0,0,0,0,Travesía a nado
1,2618,LXXXV TRAVESÍA A NADO PUERTO DE MOTRIL,2026-08-15,Motril (Granada),PRUEBA JÓVENES,lxxxv-travesia-a-nado-puerto-de-motril-prueba-...,ok,NaN,33,19,...,0,0,0,0,0,0,0,0,0,Travesía a nado
2,2628,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Roquetas de Mar (Almería),INFANTIL,travesia-100-horas-del-deporte-roquetas-de-mar...,ok,NaN,13,7,...,0,0,0,0,0,0,0,0,0,Travesía a nado
3,2628,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Roquetas de Mar (Almería),ADAPTADO INFANTIL,travesia-100-horas-del-deporte-roquetas-de-mar...,ok,NaN,3,2,...,0,0,0,0,0,0,0,0,0,Travesía a nado
4,2628,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Roquetas de Mar (Almería),ADULTOS,travesia-100-horas-del-deporte-roquetas-de-mar...,ok,NaN,49,35,...,0,0,0,0,0,0,2,0,0,Travesía a nado


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados y estado del scraping.
# "estat_esdeveniment" distingue "ok" (resultados reales), "error"
# (fallos varios, incluye los PDF pendientes de parsear) y
# "sense_resultats" — igual que "estat_esdeveniment" en cronofinisher.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print()

print("estat_esdeveniment:")
print(curses["estat_esdeveniment"].value_counts())
print()

print("De las 'error', cuántas son PDF pendientes de parsear:")
print(curses.loc[curses["estat_esdeveniment"] == "error", "error_detall"]
      .astype(str).str.startswith("pdf_pendent_de_parsejar").sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())

Valores nulos por columna:
event_id                               0
nom_cursa                              0
data                                   0
lloc                                   0
modalitat_nom                          0
modalitat_codi                       260
estat_esdeveniment                     0
error_detall                        1724
total_classificats                     0
classificat_h                          0
classificat_d                          0
classificat_total                      0
classificat_sexe_desconegut            0
DNF_h                                  0
DNF_d                                  0
DNF_total                              0
DSQ_h                                  0
DSQ_d                                  0
DSQ_total                              0
DNS_h                                  0
DNS_d                                  0
DNS_total                              0
estat_desconegut_h                     0
estat_desconegut_d            

In [3]:
# Quitamos duplicados exactos ANTES de seleccionar columnas — con "event_id"
# todavía presente. Después nos quedamos solo con "ok" (resultados reales);
# "error" (incluidos los PDF sin parsear) y "sense_resultats" no aportan
# ningún finisher.
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

antes = len(curses)
curses = curses[curses["estat_esdeveniment"] == "ok"].reset_index(drop=True)
print(f"Filas sin resultados descartadas: {antes - len(curses)} ({antes} -> {len(curses)})")

25 filas duplicadas eliminadas (1977 -> 1952)
Filas sin resultados descartadas: 260 (1952 -> 1692)


In [4]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# "lloc" viene como "Municipio (Provincia)" — la separamos con una regex
# simple (todo antes del último paréntesis / lo que hay dentro), sin
# nulos en ningún caso (comprobado más abajo). "modalitat_codi",
# DNF/DSQ/DNS y los "estat_desconegut_*" no se incluyen (no forman parte
# del esquema común y aquí no aportan nada que no den ya finisher_d/h).
curses_limpio = curses[
    ["nom_cursa", "data", "lloc", "esport", "modalitat_nom",
     "classificat_d", "classificat_h", "classificat_sexe_desconegut", "event_id"]
].rename(columns={
    "nom_cursa": "nombre_carrera",
    "data": "fecha",
    "lloc": "ubicacion",
    "modalitat_nom": "modalidad",
    "classificat_d": "finisher_d",
    "classificat_h": "finisher_h",
    "classificat_sexe_desconegut": "finisher_desconocido",
    "event_id": "id",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])

_ubicacion = curses_limpio["ubicacion"].str.extract(r"^(.*)\s\(([^)]+)\)$")
curses_limpio["municipio"] = _ubicacion[0]
curses_limpio["provincia"] = _ubicacion[1]
curses_limpio = curses_limpio.drop(columns=["ubicacion"])

curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]] = (
    curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]].fillna(0).astype(int)
)

print("Filas sin municipio/provincia extraídos:", curses_limpio["municipio"].isna().sum())
print(curses_limpio.shape)
curses_limpio.head()

Filas sin municipio/provincia extraídos: 0
(1692, 10)


,nombre_carrera,fecha,esport,modalidad,finisher_d,finisher_h,finisher_desconocido,id,municipio,provincia
0,LXXXV TRAVESÍA A NADO PUERTO DE MOTRIL,2026-08-15,Travesía a nado,PRUEBA ABSOLUTA,45,149,0,2618,Motril,Granada
1,LXXXV TRAVESÍA A NADO PUERTO DE MOTRIL,2026-08-15,Travesía a nado,PRUEBA JÓVENES,14,19,0,2618,Motril,Granada
2,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Travesía a nado,INFANTIL,6,7,0,2628,Roquetas de Mar,Almería
3,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Travesía a nado,ADAPTADO INFANTIL,1,2,0,2628,Roquetas de Mar,Almería
4,TRAVESÍA 100 HORAS DEL DEPORTE ROQUETAS DE MAR...,2026-08-15,Travesía a nado,ADULTOS,14,35,0,2628,Roquetas de Mar,Almería


In [5]:
# Clasificamos la disciplina por palabras clave en "esport" (siempre tiene
# texto en las filas "ok", a diferencia de "modalidad"). "CxM" es la
# abreviatura habitual de "Carrera por Montaña" en los circuitos de
# Andalucía (trail), y "mixta"/"Mixta (Tierra y asfalto)" implican un
# recorrido con tramo no asfaltado, así que también van a trail running.
# Natación/travesías, obstáculos, virtual y canicross van a "Otros",
# igual que en el resto de fuentes.
#
# Excepción por nombre de carrera: "Mozarabe Bike Race" es una prueba de
# ultrafondo en BTT (220 km todoterreno) pero el scraper trae "esport" =
# "Ultramaratón" (etiqueta genérica de la propia web para el listado,
# ajena a la disciplina real), y esta fuente clasifica solo por "esport",
# nunca por el nombre. No generalizamos comprobando palabras de ciclismo
# en "nombre_carrera" para todas las filas porque eso metería falsos
# positivos ya vistos en esta misma fuente (p.ej. "II MEDIA MARATÓN BTT
# SIERRA ELVIRA", que sí es una carrera a pie pese a llevar "BTT" en el
# nombre) — así que se corrige solo esta carrera concreta, por nombre.
import re

_EXCEPCIONES_NOMBRE_CICLISMO = r"mozarabe bike race"

_CATEGORIAS = {
    "Otros": r"nado|traves|obst[aá]cul|virtual|canicross|solidari",
    "Multidisciplina": r"triatl|duatl|aquatl",
    "trail running": r"trail|asfaltrail|cronoescalada|monta[ñn]a|\bcxm\b|\bmixta\b",
    "Ciclismo y btt": r"\bbtt\b|\bmtb\b|ciclis|\bbici\b|ciclotur|\bbike\b|gravel|gran ?fondo|e-?bike|ciclodeportiv",
    "marcha": r"marcha|senderismo|nordic",
    "road running": r"ruta|carrera|popular|urbana|corre|running|fondo|marat|pista|\bcross\b|\bcros\b|campo a trav[eé]s|atletismo|nocturn|circuito",
}

def _clasificar_texto(texto):
    t = "" if pd.isna(texto) else texto
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, t, flags=re.IGNORECASE):
            return categoria
    return "Otros"

def _clasificar(row):
    nombre = row["nombre_carrera"] if isinstance(row["nombre_carrera"], str) else ""
    if re.search(_EXCEPCIONES_NOMBRE_CICLISMO, nombre, flags=re.IGNORECASE):
        return "Ciclismo y btt"
    return _clasificar_texto(row["esport"])

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("Texto de 'esport' sin ninguna palabra clave reconocida (cae en Otros por defecto):")
_otros_conocidos = curses_limpio["esport"].str.contains(
    r"nado|traves|obst[aá]cul|virtual|canicross|solidari", case=False, na=False
)
print(curses_limpio.loc[(curses_limpio["tipo_modalidad"] == "Otros") & ~_otros_conocidos, "esport"].value_counts())

tipo_modalidad
road running       909
trail running      332
Ciclismo y btt     277
Otros              139
Multidisciplina     30
marcha               5
Name: count, dtype: int64

Texto de 'esport' sin ninguna palabra clave reconocida (cae en Otros por defecto):
esport
Rally            19
Ultra             2
San Silvestre     1
Varias            1
CRI               1
Infantil          1
Name: count, dtype: int64


In [6]:
# Extraemos la distancia (km) del texto de "modalidad" (K/KM, con o sin
# espacio/mayúsculas, decimales con coma o punto), con un número suelto
# sin unidad como último recurso (p.ej. "10", "7,4" — aquí, a diferencia
# de cronofinisher, un número suelto significa km, no metros: el resto de
# valores confirma que esta fuente casi siempre da la distancia ya en
# km). Maratón/media maratón se busca también en "esport" por si acaso
# "modalidad" no lo dice pero el evento sí es justo eso.
_km = curses_limpio["modalidad"].str.extract(r"(\d+(?:[.,]\d+)?)\s*[kK]", flags=re.IGNORECASE)[0]
distancia = _km.str.replace(",", ".", regex=False).astype(float)

_falta = distancia.isna()
_bare = curses_limpio["modalidad"].str.strip().str.fullmatch(r"\d{1,3}(?:[.,]\d+)?").fillna(False)
_bare_valor = curses_limpio["modalidad"].str.strip().str.replace(",", ".", regex=False)
distancia.loc[_falta & _bare] = _bare_valor.loc[_falta & _bare].astype(float)

_falta = distancia.isna()
_es_media = (
    curses_limpio["modalidad"].str.contains(r"media\s*marat", case=False, na=False)
    | curses_limpio["esport"].str.contains(r"media\s*marat", case=False, na=False)
)
_es_marat = (
    curses_limpio["modalidad"].str.contains(r"marat", case=False, na=False)
    | curses_limpio["esport"].str.contains(r"marat", case=False, na=False)
)
distancia.loc[_falta & _es_media] = 21.097
_falta = distancia.isna()
distancia.loc[_falta & _es_marat & ~_es_media] = 42.195

curses_limpio["distancia"] = distancia.fillna(0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de modalidad SIN distancia detectada:")
print(curses_limpio.loc[curses_limpio["distancia"] == 0, "modalidad"].value_counts().head(20))

Filas con distancia detectada: 1304 de 1692

Ejemplos de modalidad SIN distancia detectada:
modalidad
Varias                   255
Rally                     29
Cronoescalada              7
---                        6
Corta                      5
VARIAS                     4
250, 750, 1500 y 5000      3
ADAPTADO INFANTIL          2
Milla - 5 Millas           2
CRI + Ruta                 2
INFANTIL                   2
ADULTOS                    2
BENJAMIN                   2
--- K                      2
Milla                      2
Half                       2
ALEVIN                     2
CADETE                     2
ADAPTADO ADULTOS           2
ABSOLUTA                   2
Name: count, dtype: int64


In [7]:
# Clasificamos el público (edad) por palabras clave en "modalidad". Aquí
# solo tiene señal real de edad en las carreras extraídas por el formato
# "nuevo" de la web (categorías como "INFANTIL", "ADAPTADO ADULTOS");
# para el resto (mayoría, formato "antiguo"), "modalidad" es solo texto de
# distancia ("10 Km", "Varias") sin ninguna marca de edad, así que cae
# correctamente en Absoluta/General por defecto — no es un fallo, es que
# esa parte de la fuente no distingue categorías.
_OTROS_PATRON = r"adaptad|discapac|invident|handbike|silla de ruedas"
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

_PUBLICOS_TEXTO = {
    "Elite": r"\belite\b|profesional",
    "Mayores/Veteranos": r"veteran|master|m[aá]ster",
}
_INFANTIL_PATRON = r"infantil|beb[eé]|alev[ií]n|benjam|prebenjam|escolar|peque"
_CADETE_PATRON = r"cadete|juvenil|junior"

def _clasificar_publico(row):
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre
    if re.search(_EQUIPOS_PATRON, texto_nombre, flags=re.IGNORECASE):
        return "Equipos"

    texto = row["modalidad"]
    t = "" if pd.isna(texto) else texto
    if re.search(_EQUIPOS_PATRON, t, flags=re.IGNORECASE):
        return "Equipos"
    if re.search(_OTROS_PATRON, t, flags=re.IGNORECASE):
        return "Otros"
    for publico, patron in _PUBLICOS_TEXTO.items():
        if re.search(patron, t, flags=re.IGNORECASE):
            return publico
    if re.search(_INFANTIL_PATRON, t, flags=re.IGNORECASE):
        return "Infantil"
    if re.search(_CADETE_PATRON, t, flags=re.IGNORECASE):
        return "Cadete/Juvenil"
    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())

publico
Absoluta/General     1588
Infantil               92
Otros                   5
Cadete/Juvenil          4
Equipos                 2
Mayores/Veteranos       1
Name: count, dtype: int64


In [8]:
# "esport" ya ha cumplido su función (tipo_modalidad), lo quitamos.
curses_limpio = curses_limpio.drop(columns=["esport"])
curses_limpio.columns.tolist()

['nombre_carrera',
 'fecha',
 'modalidad',
 'finisher_d',
 'finisher_h',
 'finisher_desconocido',
 'id',
 'municipio',
 'provincia',
 'tipo_modalidad',
 'distancia',
 'publico']

### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("cruzandolameta") para identificar el origen al concatenar las tablas. `municipio`/`provincia` se separan de `lloc`; `comarca` no viene en la fuente, así que la geocodificamos a partir de `municipio`+`provincia` (igual que en buscametas/sportmaniacs). Lo que es propio solo de cruzandolameta (`finisher_desconocido`, `id`, `modalidad`) va al final.

In [9]:
# Geocodificamos "comarca" a partir de "municipio"+"provincia" (igual que
# en buscametas/sportmaniacs): aquí ya tenemos municipio y provincia
# limpios de la fuente, así que solo falta la comarca. Los eventos están
# repartidos sobre todo por Almería/Granada/Jaén, así que incluimos la
# provincia en la consulta para desambiguar. Unos pocos casos ("Cualquier
# lugar del mundo" en las carreras virtuales, "ESPAÑA" genérico) no van a
# geocodificar — es esperable, no un fallo del código.
import csv
import time


def geocodificar_comarcas(curses_limpio, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_com = out_path / "cruzandolameta_comarcas.csv"

    pares = (
        curses_limpio[["municipio", "provincia"]]
        .drop_duplicates()
        .dropna(subset=["municipio"])
    )

    cache = {}
    if csv_com.exists():
        prev = pd.read_csv(csv_com, dtype=str)
        cache = {(r["municipio"], r["provincia"]): r["comarca"] for _, r in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="cruzandolameta_comarcas_claudia")

    pendientes = [
        (m, p) for m, p in pares.itertuples(index=False)
        if (m, p) not in cache
    ]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(pares)} únicos)")

    write_header = not csv_com.exists()
    with open(csv_com, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["municipio", "provincia", "comarca"])
        if write_header:
            writer.writeheader()

        for i, (municipio, provincia) in enumerate(pendientes, 1):
            comarca = None
            try:
                query = f"{municipio}, {provincia}, España" if pd.notna(provincia) else f"{municipio}, España"
                loc = geolocator.geocode(
                    query, exactly_one=True, country_codes="es", addressdetails=True, timeout=10,
                )
                if loc:
                    comarca = loc.raw.get("address", {}).get("county")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow({"municipio": municipio, "provincia": provincia, "comarca": comarca})
            f.flush()
            cache[(municipio, provincia)] = comarca

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de comarcas: {csv_com.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/cruzandolameta")
_comarcas = geocodificar_comarcas(curses_limpio, out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio.apply(
    lambda row: _comarcas.get((row["municipio"], row["provincia"])), axis=1
)

print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "provincia", "comarca"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 207 municipios ya geocodificados


Municipios a geocodificar: 0 (de 207 únicos)
CSV de comarcas: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cruzandolameta_data\cruzandolameta_comarcas.csv
Filas con comarca: 903 de 1692


,municipio,provincia,comarca
51,Monachil,Granada,Comarca de la Vega de Granada
175,Guadix,Granada,Comarca de Guadix
79,Coín,Málaga,Valle del Guadalhorce
30,Garrucha,Almería,NaN
795,Pechina,Almería,NaN
181,Rioja,Almería,NaN
389,Santiago de Calatrava,Jaén,NaN
410,Baeza,Jaén,NaN
190,Alfacar,Granada,Comarca de la Vega de Granada
433,Ibros,Jaén,NaN


In [10]:
# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 9 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 10 fuentes, dejando lo propio de cruzandolameta (finisher_desconocido,
# id, modalidad) al final.
curses_limpio["fuente"] = "cruzandolameta"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconocido", "id", "modalidad"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconocido',
 'id',
 'modalidad']

In [11]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'finisher_desconocido', 'id', 'modalidad']
Filas x columnas: (1692, 15)

fuente                          object
nombre_carrera                  object
fecha                   datetime64[ns]
dia_semana                      object
distancia                      float64
tipo_modalidad                  object
publico                         object
finisher_d                       int64
finisher_h                       int64
municipio                       object
comarca                         object
provincia                       object
finisher_desconocido             int64
id                               int64
modalidad                       object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Cadete/Juvenil  Equipos  Infantil  \
tipo_modalidad                                        

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,finisher_desconocido,id,modalidad
1639,cruzandolameta,VII MEDIA MARATON ABADES SIERRA DE LOJA - MEMO...,2023-08-06,Domingo,21.097,Ciclismo y btt,Absoluta/General,0,6,Loja,Comarca de Loja,Granada,0,1894,Media maratón
857,cruzandolameta,SORBAS - CIRCUITO CARRERAS POPULARES DIPUTACIÓ...,2021-10-09,Sábado,8.990,road running,Absoluta/General,23,88,Sorbas,NaN,Almería,0,1364,"8,99 Km"
302,cruzandolameta,II CARRERA NOCTURNA S. SILVESTRE SANTA FE 2024,2024-12-27,Viernes,0.000,road running,Absoluta/General,43,83,Santa Fe,Comarca de la Vega de Granada,Granada,0,2215,Varias
1181,cruzandolameta,RUNNING PORT MOTRIL,2018-10-28,Domingo,5.000,road running,Absoluta/General,12,97,Motril,Comarca de la Costa Granadina,Granada,0,799,10 y 5 Km
311,cruzandolameta,32 CARRERA NOCTURNA DE GRANADA,2024-12-13,Viernes,0.000,road running,Absoluta/General,1,46,Granada,Comarca de la Vega de Granada,Granada,0,2189,Varias
875,cruzandolameta,GÁDOR - CIRCUITO CARRERAS POPULARES DIPUTACIÓN...,2021-09-18,Sábado,9.300,road running,Absoluta/General,37,135,Gádor,NaN,Almería,0,1362,"9,3"
905,cruzandolameta,CUEVAS DEL ALMANZORA - CIRCUITO CARRERAS POPUL...,2021-07-10,Sábado,6.900,road running,Absoluta/General,37,177,Cuevas del Almanzora,NaN,Almería,0,1318,"6,9K"
435,cruzandolameta,"XIV Gran Premio de Fondo ""Villa de Salobreña""",2024-04-28,Domingo,10.000,road running,Absoluta/General,214,727,Salobreña,Comarca de la Costa Granadina,Granada,0,1990,10K
560,cruzandolameta,XXVIII CARRERA URBANA ''CIUDAD DE MENGÍBAR'',2023-06-10,Sábado,1.200,road running,Absoluta/General,71,218,Mengíbar,NaN,Jaén,0,1834,"5 - 2,3 - 1,2 Km"
1022,cruzandolameta,IV CxM CASTILLO DE TAHAL,2019-11-03,Domingo,12.000,trail running,Absoluta/General,13,81,Tahal,NaN,Almería,0,1056,16 o 12 Km


In [12]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/cruzandolameta/DF_CRUZANDOLAMETA_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cruzandolameta_data\DF_CRUZANDOLAMETA_LIMPIO.csv
